In [ ]:
#Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Load Dataset
df = pd.read_csv('Titanic-Dataset.csv')

# Preprocessing: Drop columns with too many missing values or redundant info
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# Fill missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Encode Categorical Data
le_sex = LabelEncoder()
le_emb = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex'])
df['Embarked'] = le_emb.fit_transform(df['Embarked'])

# Features and Target
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

AttributeError: module 'pandas' has no attribute 'load_dataset'

In [ ]:
#2. Systematic Hyperparameter Tuning

In [ ]:
#A. Exhaustive Search: GridSearchCV

In [ ]:
# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                           param_grid=param_grid,
                           cv=5, # 5-Fold Cross-Validation
                           n_jobs=-1,
                           verbose=1)

grid_search.fit(X_train, y_train)

print(f"Best Parameters (GridSearch): {grid_search.best_params_}")
print(f"Best Score: {grid_search.best_score_:.4f}")

In [ ]:
#B. Randomized Search: RandomizedSearchCV

In [ ]:
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_split': randint(2, 11),
    'bootstrap': [True, False]
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                                   param_distributions=param_dist,
                                   n_iter=20, # Number of parameter settings sampled
                                   cv=5,
                                   random_state=42,
                                   n_jobs=-1)

random_search.fit(X_train, y_train)

print(f"Best Parameters (RandomSearch): {random_search.best_params_}")

In [ ]:
#Robust Evaluation (K-Fold Cross-Validation)

In [ ]:
best_rf = grid_search.best_estimator_

# Perform 10-Fold Cross Validation
cv_scores = cross_val_score(best_rf, X, y, cv=10)

print(f"CV Mean Accuracy: {np.mean(cv_scores):.4f}")
print(f"CV Standard Deviation: {np.std(cv_scores):.4f}")

# Evaluate on test set
test_score = best_rf.score(X_test, y_test)
print(f"\nTest Set Accuracy: {test_score:.4f}")